In [ ]:
# Install PyTorch 2.3.0 + CUDA 11.8 (compatible with FlashAttention source)
!pip install torch==2.3.0+cu118 torchvision==0.18.0+cu118 torchaudio==2.3.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Install helper libraries
!pip install numpy einops
!pip install ninja  # required for PyTorch C++/CUDA extensions


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 863.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.makedirs("flash_attn_custom", exist_ok=True)


In [ ]:
%%writefile attention.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void flash_attn_kernel(const float* q, const float* k, const float* v, float* out, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        out[idx] = q[idx] * k[idx] + v[idx];  // Simplified example
    }
}

torch::Tensor flash_attn_forward(torch::Tensor q, torch::Tensor k, torch::Tensor v) {
    auto out = torch::zeros_like(q);
    int threads = 256;
    int blocks = (q.numel() + threads - 1) / threads;
    flash_attn_kernel<<<blocks, threads>>>(q.data_ptr<float>(), k.data_ptr<float>(), v.data_ptr<float>(), out.data_ptr<float>(), q.numel());
    return out;
}


Overwriting attention.cu


In [ ]:
%%writefile binding.cpp
#include <torch/extension.h>

// Declare the function from attention.cu
torch::Tensor flash_attn_forward(torch::Tensor q, torch::Tensor k, torch::Tensor v);

PYBIND11_MODULE(flash_attn_cuda, m) {
    m.def("flash_attn", &flash_attn_forward, "FlashAttention forward kernel");
}


Writing binding.cpp


In [ ]:
from torch.utils.cpp_extension import load

flash_attn_cuda = load(
    name="flash_attn_cuda",
    sources=["attention.cu", "binding.cpp"],
    verbose=True,
    extra_cuda_cflags=["--expt-relaxed-constexpr"]
)


Using /root/.cache/torch_extensions/py312_cu118 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu118/flash_attn_cuda/build.ninja...
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module flash_attn_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module flash_attn_cuda...


In [ ]:
import torch
import time

# Dummy data
N, D = 1024*1024, 1  # large vector for demo
q = torch.randn(N, device='cuda', dtype=torch.float32)
k = torch.randn(N, device='cuda', dtype=torch.float32)
v = torch.randn(N, device='cuda', dtype=torch.float32)

# Benchmark FlashAttention
torch.cuda.synchronize()
start = time.time()
out_flash = flash_attn_cuda.flash_attn(q, k, v)
torch.cuda.synchronize()
flash_time = time.time() - start
print(f"FlashAttention kernel time: {flash_time:.6f} s")

# Benchmark PyTorch native attention (simplified)
torch.cuda.synchronize()
start = time.time()
out_native = q * k + v  # mimic attention computation
torch.cuda.synchronize()
native_time = time.time() - start
print(f"PyTorch native computation time: {native_time:.6f} s")

# Speedup
print(f"Speedup: {native_time / flash_time:.2f}x")


FlashAttention kernel time: 0.006691 s
PyTorch native computation time: 0.013464 s
Speedup: 2.01x
